# Process large files

You need to process a text file that is too large to fit comfortably in memory. Loading the entire file with `read()` or `readlines()` would consume too much memory or cause your program to crash.

This guide covers several approaches for processing large files efficiently.

## Create a sample file for testing

First, create a sample file with many lines to demonstrate the techniques.

In [ ]:
import os
from pathlib import Path

lines = [
    f"Line {i}: This is sample data for testing large file processing.\n"
    for i in range(1, 10001)
]
Path("large-sample.txt").write_text("".join(lines), encoding="utf-8")

size_mb = os.path.getsize("large-sample.txt") / (1024 * 1024)
print(f"Created large-sample.txt ({size_mb:.2f} MB, {len(lines)} lines)")

## Solution 1: iterate line by line

The simplest and most common approach is to iterate directly over the file object. This reads one line at a time, keeping memory usage constant regardless of file size.

In [ ]:
from pathlib import Path

count = 0
with Path("large-sample.txt").open("r", encoding="utf-8") as f:
    for line in f:
        count += 1

print(f"Processed {count} lines")

## Solution 2: read in fixed-size chunks

For files where you do not need line-by-line processing, you can read in fixed-size chunks. This is useful for character counting, binary processing, or when line boundaries do not matter.

In [ ]:
from pathlib import Path


def process_in_chunks(filepath: str | Path, chunk_size: int = 8192) -> int:
    """Count characters in a file by reading in chunks.

    Args:
        filepath: The path to the file to process.
        chunk_size: The number of characters to read at a time.

    Returns:
        The total number of characters in the file.
    """
    path = Path(filepath)
    total_chars = 0
    with path.open("r", encoding="utf-8") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            total_chars += len(chunk)
    return total_chars


result = process_in_chunks("large-sample.txt")
print(f"Total characters: {result:,}")

## Solution 3: use a generator for lazy processing

Generators process data lazily -- they only read and process data as it is requested. This is ideal for filtering or transforming lines without loading everything into memory.

In [ ]:
from collections.abc import Iterator
from pathlib import Path


def filtered_lines(filepath: str | Path, keyword: str) -> Iterator[str]:
    """Yield lines from a file that contain a given keyword.

    Args:
        filepath: The path to the file to search.
        keyword: The text to search for in each line.

    Yields:
        Lines that contain the keyword, with whitespace stripped.
    """
    path = Path(filepath)
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if keyword in line:
                yield line.strip()


matches = list(filtered_lines("large-sample.txt", "Line 500:"))
for match in matches:
    print(match)

## Solution 4: process large CSV files

The `csv.reader()` and `csv.DictReader()` functions process CSV files row by row, so they are naturally memory-efficient.

In [ ]:
import csv
from pathlib import Path

csv_lines = ["name,value\n"] + [
    f"item_{i},{i * 1.5}\n" for i in range(1, 1001)
]
Path("large-data.csv").write_text("".join(csv_lines), encoding="utf-8")

total = 0.0
count = 0
with open("large-data.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total += float(row["value"])
        count += 1

print(f"Processed {count} rows, total value: {total:.2f}")

## Alternative approaches

For more specialised use cases, consider the following:

- The `mmap` module for memory-mapped file access (random access without loading the entire file)
- The `linecache` module for random access to specific lines
- Third-party libraries such as `pandas` (with the `chunksize` parameter) for structured data processing
- The `multiprocessing` module for parallel processing of CPU-bound tasks

## Key takeaways

- Iterate line by line for most text processing tasks
- Use fixed-size chunks for binary or character-level processing
- Use generators for lazy filtering and transformation
- `csv.reader()` and `csv.DictReader()` process CSV files row by row efficiently
- Always use `with` statements to ensure files are closed properly

In [ ]:
from pathlib import Path

Path("large-sample.txt").unlink(missing_ok=True)
Path("large-data.csv").unlink(missing_ok=True)
print("Temporary files removed.")